In [66]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import random

# ---------------------------
# 1. DATASET
# ---------------------------
users = [
    'Cesar','Citlali','Salvador','Daniel','Martin',
    'Michelle','Angela','Kevin','Sucem','Fabian',
    'Francisco','Jim','Ximena','Erika','Alan','Omar'
]

movies = [
    'Harry Potter','Infinity War','Exorcista','Interestelar',
    'Viaje de chihiro','Toy story 2','Mi amigo robot','Bastardos sin gloria',
    'Spider man miles','Guerra mundial Z','Shrek 2',
    'End game','Tumba de luciernagas','La empleada','Karate kid'
]

data = []

for user in users:
    for movie in movies:
        if random.random() > 0.3:  
            if user == "Cesar" and movie == "Harry Potter":
                rating = 5
            else:
                rating = random.randint(1,5)

            data.append({
                'user_id': user,
                'movie_title': movie,
                'rating': rating
            })

df = pd.DataFrame(data)
df.to_csv("peliculas.csv", index=False)

# ---------------------------
# 2. MATRIZ
# ---------------------------
matrix = df.pivot_table(index='user_id', columns='movie_title', values='rating').fillna(0)

# ---------------------------
# 3. SIMILITUD
# ---------------------------
sim = cosine_similarity(matrix.T)
df_sim = pd.DataFrame(sim, index=matrix.columns, columns=matrix.columns)

# ---------------------------
# 4. RECOMENDADOR 
# ---------------------------
def recommend(user):
    user_ratings = matrix.loc[user]
    watched = user_ratings[user_ratings > 0]

    scores = {}

    for movie in watched.index:
        for m in df_sim.columns:
            if user_ratings[m] == 0:
                scores[m] = scores.get(m, 0) + df_sim[movie][m] * watched[movie]

    if len(scores) == 0:
        scores = df.groupby('movie_title')['rating'].mean()
        scores = scores[~scores.index.isin(watched.index)]

    scores = pd.Series(scores).sort_values(ascending=False)

    fav = watched.idxmax()

    return fav, scores.head(6)

# ---------------------------
# 5. UI
# ---------------------------
dropdown = widgets.Dropdown(
    options=users,
    description='Perfil:'
)

output = widgets.Output()

def update(change):
    with output:
        clear_output()
        user = dropdown.value
        fav, recs = recommend(user)
        
        display(HTML(f"""
        <div style="
            background:#141414;
            color:white;
            padding:20px;
            border-radius:12px;
            font-family:Arial;
        ">
            <h1 style="color:#088558;">Tarea 1: Recomendación de Películas</h1>

            <h3>Usuario: {user}</h3>
            <h2>Favorita: {fav}</h2>

            <h3 style="margin-top:20px;">Recomendadas para ti</h3>

            <div style="
                display:flex;
                gap:15px;
                overflow-x:auto;
                padding:10px 0;
            ">
                {''.join([f'''
                    <div style="
                        min-width:150px;
                        height:200px;
                        background:#222;
                        border-radius:10px;
                        display:flex;
                        align-items:center;
                        justify-content:center;
                        text-align:center;
                        padding:10px;
                        font-weight:bold;
                    ">
                        {movie}
                    </div>
                ''' for movie in recs.index]) if not recs.empty else "<p>No hay recomendaciones</p>"}
            </div>

            <h3 style="margin-top:30px;">Gráficas</h3>
        </div>
        """))

        # ---------------------------
        # GRÁFICAS HORIZONTALES
        # ---------------------------
        plt.style.use('dark_background')

        fig, axes = plt.subplots(1, 2, figsize=(14,5))
        fig.patch.set_facecolor('#141414')

        # -------- Gráfica 1 --------
        axes[0].set_facecolor('#141414')

        df.groupby('movie_title')['rating'].mean().sort_values(ascending=False)\
            .plot(kind='bar', ax=axes[0])

        axes[0].set_title("Top películas", color='white')
        axes[0].tick_params(colors='white')
        axes[0].tick_params(axis='x', rotation=45)

        # -------- Gráfica 2 --------
        axes[1].set_facecolor('#141414')

        if not recs.empty:
            recs.plot(kind='bar', ax=axes[1])
        else:
            axes[1].text(0.5, 0.5, 'Sin recomendaciones',
                         color='white', ha='center', va='center')

        axes[1].set_title(f"Recomendaciones para {user}", color='white')
        axes[1].tick_params(colors='white')
        axes[1].tick_params(axis='x', rotation=45)

        plt.tight_layout()
        plt.show()

dropdown.observe(update, names='value')

# ---------------------------
# 6. MOSTRAR
# ---------------------------
display(dropdown, output)
update(None)

Dropdown(description='Perfil:', options=('Cesar', 'Citlali', 'Salvador', 'Daniel', 'Martin', 'Michelle', 'Ange…

Output()